In [0]:
%sql
USE CATALOG fmcg;
CREATE SCHEMA IF NOT EXISTS abac_demo;
USE SCHEMA abac_demo ;
CREATE TABLE IF NOT EXISTS customer_details (
    customer_id   INT PRIMARY KEY,
    name          VARCHAR(100) NOT NULL,
    email         VARCHAR(150) UNIQUE NOT NULL,
    phone         VARCHAR(40),
    address       VARCHAR(255),
    ssn           CHAR(11) UNIQUE,
    region        VARCHAR(50)
) USING DELTA ; 

In [0]:
%sql
ALTER TABLE abac_demo.customer_details
CHANGE COLUMN phone phone VARCHAR(40);


In [0]:
%sql
INSERT INTO customer_details (customer_id, name, email, phone, address, ssn, region) VALUES
(1, 'Alice Johnson', 'alice.johnson@usmail.com', '+1-202-555-0101', '123 Main St, New York, NY', '123-45-6789', 'US'),
(2, 'Rajesh Kumar', 'rajesh.kumar@asiamail.com', '+91-9876543210', '45 MG Road, Bengaluru, India', '234-56-7890', 'Asia'),
(3, 'Sophie Müller', 'sophie.muller@europemail.com', '+49-151-2345678', '12 Hauptstrasse, Berlin, Germany', '345-67-8901', 'Europe'),
(4, 'Carlos Silva', 'carlos.silva@latamail.com', '+55-21-98765432', 'Av Paulista, São Paulo, Brazil', '456-78-9012', 'Other'),
(5, 'Emily Davis', 'emily.davis@usmail.com', '+1-303-555-0199', '456 Elm St, Denver, CO', '567-89-0123', 'US'),
-- … continue pattern with varied names, emails, phones, addresses, SSNs, and regions …
(50, 'Hiroshi Tanaka', 'hiroshi.tanaka@asiamail.com', '+81-90-1234-5678', 'Shibuya, Tokyo, Japan', '987-65-4321', 'Asia');

In [0]:
%sql
SELECT * FROM abac_demo.customer_details; 

In [0]:
%sql
-- APPLY PII TAGS 
ALTER TABLE fmcg.abac_demo.customer_details
ALTER COLUMN ssn SET TAGS ("pii" = "ssn") ;

ALTER TABLE fmcg.abac_demo.customer_details
ALTER COLUMN email  SET TAGS ("pii" = "email") ;

ALTER TABLE fmcg.abac_demo.customer_details 
ALTER COLUMN address SET TAGS ("pii" = "address") ; 

ALTER TABLE fmcg.abac_demo.customer_details
ALTER COLUMN phone SET TAGS ("pii" = "phone");

ALTER TABLE fmcg.abac_demo.customer_details
ALTER COLUMN region SET TAGS ("pii" = "region")

In [0]:
%sql
CREATE OR REPLACE FUNCTION is_not_usregion(region STRING) 
RETURNS boolean 
RETURN (
    CASE WHEN region LIKE '%US%' THEN false
    ELSE true
    END
) ; 


-- CREATE OR REPLACE FUNCTION fmcg.abac_demo.is_not_usregion(address STRING)
-- RETURNS BOOLEAN
-- RETURN CASE WHEN address LIKE '%US%' THEN FALSE ELSE TRUE END;

-- email masking 
CREATE OR REPLACE FUNCTION email_masking(email STRING)
RETURNS STRING
RETURN CONCAT('****@', SPLIT(email,'@')[1]) ;

--ssn masking 
CREATE OR REPLACE FUNCTION ssn_masking_half(ssn STRING)
RETURNS STRING
RETURN CONCAT('XXX-XX',substr(ssn,8,4)) ;

--ssn masking full 
CREATE OR REPLACE FUNCTION ssn_masking_full(ssn STRING)
RETURNS STRING
RETURN 'XXX-XX-XXXX' ; 


--phone masking 
CREATE OR REPLACE FUNCTION phone_masking(phone STRING)
RETURNS STRING
RETURN CONCAT('XXX-XXX-',substr(phone,9,4)) ;

-- phone number masking partitial 
CREATE OR REPLACE FUNCTION mask_partial_phone(phone STRING)
RETURNS STRING
RETURN CONCAT(
    SUBSTRING(phone, 1, 3),              
    SHA2(SUBSTRING(phone, 4), 256)     
);


In [0]:
%sql
-- ROW FILTER POLICY 

-- 2. Create the ABAC row-filter policy
CREATE POLICY filter_non_us_region
ON SCHEMA fmcg.abac_demo
COMMENT 'Filter out non US region customers'
ROW FILTER is_not_usregion
TO `souviksarkar1000@gmail.com`
FOR TABLES
MATCH COLUMNS has_tag_value('pii', 'region') AS region_col
USING COLUMNS (region_col);


-- COLUMN MASKING POLICY 
CREATE POLICY mask_email
ON SCHEMA fmcg.abac_demo 
COMMENT " Mask email address "
COLUMN MASK  email_masking 
TO `souviksarkar1000@gmail.com`
FOR TABLES 
MATCH COLUMNS has_tag_value('pii', 'email') as email_col 
ON COLUMN email_col ;


CREATE POLICY mask_phone
ON SCHEMA fmcg.abac_demo
COMMENT 'Mask phone number'
COLUMN MASK fmcg.abac_demo.mask_partial_phone
TO `souviksarkar1000@gmail.com`
FOR TABLES
MATCH COLUMNS has_tag_value('pii', 'phone') AS phone_col
ON COLUMN phone_col;

CREATE POLICY mask_ssn
ON SCHEMA fmcg.abac_demo 
COMMENT " Mask ssn "
COLUMN MASK ssn_masking_half 
TO `souviksarkar1000@gmail.com`
FOR TABLES 
MATCH COLUMNS has_tag_value('pii', 'ssn') as ssn_col 
ON COLUMN ssn_col ;


In [0]:
%sql
CREATE POLICY filter_non_us_region_new
ON SCHEMA fmcg.abac_demo
COMMENT 'Filter out non US region customers'
ROW FILTER fmcg.abac_demo.is_not_usregion
TO `souviksarkar1000@gmail.com`
FOR TABLES
MATCH COLUMNS has_tag_value('pii', 'region') AS region_col
USING COLUMNS (region_col);

In [0]:
%sql
DROP POLICY filter_non_us_region
ON SCHEMA fmcg.abac_demo;

In [0]:
%sql
--SHOW POLICIES ON SCHEMA fmcg.abac_demo;
DROP POLICY filter_non_usregion ON SCHEMA fmcg.abac_demo ;

In [0]:
%sql
SELECT * FROM fmcg.abac_demo.customer_details